# 05 — Phase 1b: Round 1 Evaluation — LLM Judge

First evaluation run: 3 exercises per version (9 total). Judge sees NARRATIVE + DATA + TASK only (TOPIC and CONSOLIDATION NOTE stripped before scoring).

## 1. Setup

In [1]:
import sys
from pathlib import Path
import importlib
import re

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.utils.llm_client import LLMClient
import src.generation.pf_generator as pfgen
from src.generation.pf_generator import PFGenerator

# Swiss AI — Llama-3.3-70B for both generation and judging
gen_client = LLMClient.from_role("structured")
judge_client = LLMClient.from_role("structured")

gen = PFGenerator(gen_client)
print(f"Generator: {gen_client.model}")
print(f"Judge:     {judge_client.model}")
print("Ready ✓")

Generator: meta-llama/Llama-3.3-70B-Instruct
Judge:     meta-llama/Llama-3.3-70B-Instruct
Ready ✓


## 2. Generate 3 exercises per version (9 total)

In [2]:
N_PER_VERSION = 3
exercises = []

for i in range(N_PER_VERSION):
    for version, generate_fn in [("V1", gen.generate_v1), ("V2", gen.generate_v2), ("V3", gen.generate_v3)]:
        ex = generate_fn(grade="7th grade", topic="ratios & proportional relationships", temperature=0.8)
        ex.version = version  # ensure correct label
        exercises.append(ex)
        print(f"[{version}] #{i+1}: valid={ex.is_valid()} | {ex.sections.get('TOPIC', 'N/A')[:50]}")

print(f"\nTotal: {len(exercises)} exercises ({N_PER_VERSION} × 3 versions)")


[V1] #1: valid=True | Ratios & proportional relationships, recipe scalin
[V2] #1: valid=True | Ratios & proportional relationships, baking cookie
[V3] #1: valid=True | 7th grade, ratios & proportional relationships, tr
[V1] #2: valid=True | Ratios & proportional relationships, baking cookie
[V2] #2: valid=True | Ratios & proportional relationships, trail mix rec
[V3] #2: valid=True | Ratios & proportional relationships, scaling recip
[V1] #3: valid=True | Ratios & proportional relationships, music festiva
[V2] #3: valid=True | Ratios and proportional relationships, small busin
[V3] #3: valid=True | Ratios & proportional relationships, school cafete

Total: 9 exercises (3 × 3 versions)


## 3. Judge prompt — TOPIC stripped, only student-facing content

In [5]:
JUDGE_SYSTEM = """You are an evaluator of Productive Failure (PF) exercises for mathematics.
For each dimension below, FIRST analyze what you observe, THEN assign a score of 1, 2, or 3.

SCORING GUIDE — most exercises score 2. Reserve 3 for genuinely exceptional quality.

1. Activation of Prior Knowledge (APK)
   Analyze: What prerequisite concepts does this exercise require? Are they genuinely needed to attempt the problem, or merely present in the background?
   Score 1: No prerequisite concepts; exercise is self-contained.
   Score 2: One prerequisite concept is genuinely needed (e.g., unit rates).
   Score 3: Two or more interacting prerequisites are essential (e.g., ratio reasoning + unit conversion).

2. Sweet Spot Calibration (SSC)  
   Analyze: What would a student using only prior knowledge produce? Would their answer look reasonable at first glance? Is the trap structural (built into the data) or superficial (just an instruction to avoid a mistake)?
   Score 1: Too easy (prior knowledge trivially solves) or too hard (unsolvable without instruction).
   Score 2: Moderate challenge; prior knowledge makes partial progress but stalls.
   Score 3: Prior knowledge produces a specific, plausible-looking but WRONG answer. The trap is structural — the data is deliberately rigged.

3. Affective Engagement (AE)
   Analyze: Are there named characters with real stakes? Would a student care about solving this, or is it just a disguised worksheet problem?
   Score 1: Abstract or textbook-like; no relatable context.
   Score 2: Real-world scenario present, some context.
   Score 3: Named, relatable characters facing a genuine situation with stakes — the problem feels like a story, not an assignment.

4. Multiple Representations (MR)
   Analyze: Count the distinct representation types in the exercise (narrative text, data table, diagram reference, equation, etc.).
   Score 1: Text-only; single format.
   Score 2: Two representation types.
   Score 3: Three or more types.

5. Open-Endedness (OE)
   Analyze: Does this exercise have a single correct answer, or genuinely multiple solution pathways? Could different students produce meaningfully different answers? Are verbs like "design" and "invent" backed by a task that truly admits multiple approaches, or are they just window dressing on a computation problem?
   Score 1: Single answer, single path; "solve for X" style.
   Score 2: Some flexibility; limited interpretation possible.
   Score 3: Multiple genuine pathways; different students would produce meaningfully different solutions. The task cannot be reduced to "find the right number."

OUTPUT exactly in this format:
APK ANALYSIS: [your analysis]
APK: [1-3]
SSC ANALYSIS: [your analysis]  
SSC: [1-3]
AE ANALYSIS: [your analysis]
AE: [1-3]
MR ANALYSIS: [your analysis]
MR: [1-3]
OE ANALYSIS: [your analysis]
OE: [1-3]
TOTAL: [sum]/15"""

def strip_topic(text: str) -> str:
    """Remove the TOPIC section from the exercise text before judging."""
    text = re.sub(r'(?:^|\n)\s*(?:#+\s*)?(?:\d+[\.\)]\s*)?TOPIC[:\s].*?(\n|$)', '\n', text, flags=re.IGNORECASE)
    text = re.sub(r'TOTAL:\s*\d+\s*/\s*15\b.*', '', text)
    return text.strip()

# Test strip on first exercise
print("=== BEFORE STRIP ===")
print(exercises[0].raw_output[:250])
print("\n=== AFTER STRIP ===")
print(strip_topic(exercises[0].raw_output)[:250])

=== BEFORE STRIP ===
## TOPIC
Ratios & proportional relationships, recipe scaling in a bakery

## NARRATIVE
Maria's Bakery is famous for its delicious chocolate cake. The secret recipe requires a specific ratio of flour, sugar, and cocoa powder. The original recipe makes

=== AFTER STRIP ===
## NARRATIVE
Maria's Bakery is famous for its delicious chocolate cake. The secret recipe requires a specific ratio of flour, sugar, and cocoa powder. The original recipe makes 12 cakes and uses 480 grams of flour, 270 grams of sugar, and 180 grams o


## 4. Evaluate all 9 exercises (blinded, shuffled, temp=0, TOPIC stripped)

In [6]:
import random

# Shuffle to prevent order effects
eval_order = list(range(len(exercises)))
random.shuffle(eval_order)

results = []
for idx in eval_order:
    ex = exercises[idx]
    exercise_text = strip_topic(ex.raw_output)
    
    messages = [
        {"role": "system", "content": JUDGE_SYSTEM},
        {"role": "user", "content": exercise_text},
    ]
    resp = judge_client.generate(messages, max_tokens=1024, temperature=0.0)
    
    # Parse scores — look for "DIM: [1-3]" pattern (skip ANALYSIS lines)
    scores = {}
    for dim in ["APK", "SSC", "AE", "MR", "OE"]:
        m = re.search(rf'{dim}:\s*(\d)', resp.text)
        scores[dim] = int(m.group(1)) if m else 0
    total_m = re.search(r'TOTAL:\s*(\d+)', resp.text)
    scores["TOTAL"] = int(total_m.group(1)) if total_m else sum(scores.values())
    
    results.append({
        "version": ex.version,
        **scores,
    })
    print(f"[{len(results)}/9] {ex.version} | APK={scores['APK']} SSC={scores['SSC']} AE={scores['AE']} MR={scores['MR']} OE={scores['OE']} | TOTAL={scores['TOTAL']}/15")

import pandas as pd
df = pd.DataFrame(results)
print("\nDone ✓")

[1/9] V3 | APK=3 SSC=3 AE=3 MR=2 OE=2 | TOTAL=13/15
[2/9] V1 | APK=3 SSC=3 AE=3 MR=3 OE=3 | TOTAL=15/15
[3/9] V1 | APK=3 SSC=3 AE=3 MR=3 OE=3 | TOTAL=15/15
[4/9] V2 | APK=3 SSC=3 AE=3 MR=3 OE=3 | TOTAL=15/15
[5/9] V1 | APK=3 SSC=3 AE=3 MR=3 OE=3 | TOTAL=15/15
[6/9] V2 | APK=2 SSC=3 AE=2 MR=2 OE=2 | TOTAL=11/15
[7/9] V3 | APK=3 SSC=3 AE=3 MR=3 OE=3 | TOTAL=15/15
[8/9] V3 | APK=3 SSC=3 AE=2 MR=3 OE=3 | TOTAL=13/15
[9/9] V2 | APK=3 SSC=3 AE=3 MR=3 OE=3 | TOTAL=15/15

Done ✓


## 5. Results — per-version means

In [7]:
# Per-version means
summary = df.groupby("version")[["APK", "SSC", "AE", "MR", "OE", "TOTAL"]].mean().round(1)
print(summary.to_string())

print("\n--- Per-dimension diff vs V1 ---")
v1_means = df[df["version"]=="V1"][["APK","SSC","AE","MR","OE","TOTAL"]].mean()
for v in ["V2", "V3"]:
    vx_means = df[df["version"]==v][["APK","SSC","AE","MR","OE","TOTAL"]].mean()
    diff = vx_means - v1_means
    print(f"{v} - V1: {dict(diff.round(1))}")

print("\n--- Raw scores ---")
print(df.to_string(index=False))

         APK  SSC   AE   MR   OE  TOTAL
version                                
V1       3.0  3.0  3.0  3.0  3.0   15.0
V2       2.7  3.0  2.7  2.7  2.7   13.7
V3       3.0  3.0  2.7  2.7  2.7   13.7

--- Per-dimension diff vs V1 ---
V2 - V1: {'APK': np.float64(-0.3), 'SSC': np.float64(0.0), 'AE': np.float64(-0.3), 'MR': np.float64(-0.3), 'OE': np.float64(-0.3), 'TOTAL': np.float64(-1.3)}
V3 - V1: {'APK': np.float64(0.0), 'SSC': np.float64(0.0), 'AE': np.float64(-0.3), 'MR': np.float64(-0.3), 'OE': np.float64(-0.3), 'TOTAL': np.float64(-1.3)}

--- Raw scores ---
version  APK  SSC  AE  MR  OE  TOTAL
     V3    3    3   3   2   2     13
     V1    3    3   3   3   3     15
     V1    3    3   3   3   3     15
     V2    3    3   3   3   3     15
     V1    3    3   3   3   3     15
     V2    2    3   2   2   2     11
     V3    3    3   3   3   3     15
     V3    3    3   2   3   3     13
     V2    3    3   3   3   3     15


## Observation — judge ceiling effect & literature-backed fix

**Problem:** 8/9 exercises scored 15/15. The judge has near-zero discrimination — it cannot tell the difference between a genuinely open-ended PF exercise and a ratio word problem with "design" as the verb. Five likely causes:

1. No calibration baseline — judge sees one exercise at a time, no reference distribution
2. "Expert" frame primes cheerleading, not criticism
3. Structure is conflated with quality — all exercises have proper formatting, so the judge assumes content is good
4. 3-point scale too coarse without concrete anchors
5. No counterfactual reasoning — judge can't ask "what would make this a 2?"

**Literature-backed fix — G-Eval (Liu et al., 2023):**

Liu et al. showed that adding a **Chain-of-Thought analysis step before each score** improves Spearman correlation with human judgments from ~0.3 to 0.514. The LLM must first reason about the criterion, then assign a score — the reasoning step forces calibration.

Additionally, Zheng et al. (2023) found that **reference examples** with known scores mitigate position bias and score inflation, though this requires pre-scored calibration exercises.

**Next:** Implement the G-Eval CoT approach — replace the judge prompt's "score + reason" format with "analyze → score" for each dimension.

### References

- Liu, Y., Iter, D., Xu, Y., Wang, S., Xu, R., & Zhu, C. (2023). G-Eval: NLG Evaluation using GPT-4 with Better Human Alignment. *EMNLP 2023*.
- Zheng, L., Chiang, W.-L., Sheng, Y., et al. (2023). Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena. *NeurIPS 2023 Datasets and Benchmarks Track*.
- Wang, J., Liang, Y., Meng, F., et al. (2023). Is ChatGPT a Good NLG Evaluator? A Preliminary Study. *NewSumm@EMNLP 2023*.